In [ ]:
"""
adding notagen corpus
"""

from music21 import *
notagencorpus = corpus.corpora.LocalCorpus("notagen")
notagencorpus.addPath(r"~\notagen-pieces\scores\Baroque_BachJohannSebastian_Choral")
notagencorpus.save()
for path in notagencorpus.getPaths():
    score = corpus.parse(path) # is that a way to iterate over our notagen corpus?

In [ ]:
"""
1. Keys:
Algorithms: determine key of chorale by applying music21's key finding algorithm.
- count how often each key occurs, e.g.
bach_keys = { "C": 23, "C#": 1. "D": 12, ..."c": 32, "c#": 2, ...}
notagen_keys = { "C": 58, "C#": 1. "D": 6, ..."c": 14, "c#": 2, ...}

2. Time signatures:
Algorithm: take first time signature
bach_tss = { "4/4": 23, "3/4": 1. "5/8": 0, ...}
notagen_tss = { "4/4": 58, "3/4": 1. "5/8": 1, ...}

3. Melodies:
Algorithm:
  - transpose everything to C major / a minor (Bach AND Notagen)
  - take only 10 first notes of Soprano

4. Dominant seventh chords
...

5. Parallel Fiths (Google search)

6. Dv / Neapolitaner
...

7. Auftakte / Volltakte
"""

In [ ]:
"""
J:
1. Keys:
Algorithms: determine key of chorale by applying music21's key finding algorithm.
- count how often each key occurs, e.g.
bach_keys = { "C": 23, "C#": 1. "D": 12, ..."c": 32, "c#": 2, ...}
notagen_keys = { "C": 58, "C#": 1. "D": 6, ..."c": 14, "c#": 2, ...}
"""
import music21 as m21
from collections import Counter
keys_BACH = []
keys_notagen = []

# BACHkeys
bach_chorales = m21.corpus.chorales.Iterator()
for c in bach_chorales:
    chorales_key = c.analyze("key")
    keys_BACH.append(f"{chorales_key.tonic.name} {chorales_key.mode}")
keys_total = Counter(keys_BACH)
bach_keys = sorted(keys_total.items()) # this casts keys_total to a list!

# NOTAGENkeys
for path in notagencorpus.getPaths():
    score = corpus.parse(path)
    ai_keys = score.analyze("key")
    keys_notagen.append(f"{ai_keys.tonic.name} {ai_keys.mode}")
ai_keys_total = Counter(keys_notagen)
ai_keys = sorted(ai_keys_total.items())

# output
print("BACH KEYS:")
print(*bach_keys, sep="\n")
print("-----------------")
print("NOTAGEN KEYS:")
print(*ai_keys, sep="\n")



In [31]:
"""
2. Time signatures:
Algorithm: take first time signature
bach_tss = { "4/4": 23, "3/4": 1. "5/8": 0, ...}
notagen_tss = { "4/4": 58, "3/4": 1. "5/8": 1, ...}
"""
""" NOTAGEN CHORALES; Iteration doesn't work
from collections import Counter
notagen_tss_list = []
notagen_chorales = m21.corpus.corpora.LocalCorpus("ai-bach").Iterator() #?
for piece in notagen_chorales:
  time_sig = piece.recurse().getElementsByClass(m21.meter.TimeSignature)[0]
  notagen_tss_list.append(time_sig.ratioString)
notagen_tss = Counter(notagen_tss_list)
print(notagen_tss)
"""
from collections import Counter


# BACH

bach_tss_list = []
bach_chorales = m21.corpus.chorales.Iterator()
for piece in bach_chorales:
    time_sig = piece.recurse().getElementsByClass(m21.meter.TimeSignature)[0]
    bach_tss_list.append(time_sig.ratioString)
bach_tss = Counter(bach_tss_list)
print(f"BACH TIME SIGNATURES: {bach_tss}")

# NOTAGEN
notagen_tss_list = []
for path in notagencorpus.getPaths():
    score = corpus.parse(path)
    time_sig = score.recurse().getElementsByClass(m21.meter.TimeSignature)[0]
    notagen_tss_list.append(time_sig.ratioString)
notagen_tss_list = Counter(notagen_tss_list)
print(f"NOTAGEN TIME SIGNATURES: {notagen_tss_list}")

    






BACH TIME SIGNATURES: Counter({'4/4': 331, '3/4': 38, '3/2': 1, '12/8': 1})
NOTAGEN TIME SIGNATURES: Counter({'4/4': 100})


In [ ]:
"""
3. Melodies:
Algorithm:
  - transpose everything to C major / a minor (Bach AND Notagen)
  - take only 10 first notes of Soprano
"""
import music21 as m21
def extract_snippets(corpus=None):
  if corpus is not None:
    snippets = {}
    for chorale in corpus:
      try:
        ## transpose to C
        key = chorale.analyze("key")
        if key.mode == "major":
          interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("C"))
        elif key.mode == "minor":
          interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("A"))
        else:
          print("error")

        transposed = chorale.transpose(interval)
        for part in chorale.parts:
          if part.partName == "Soprano": # does that work? We should do smell-tests
            id = transposed.metadata.movementName
            soprano = transposed.recurse().notes # <--- we needed recurse instead of flatten! i dont know why, but with flatten the transposed snippets were not right
            notes = [note.pitch.nameWithOctave for note in soprano.notes][:10]
            snippets[id] = notes
          else:
            continue
      except:
        pass

    return snippets
  else:
    return None

bach_chorales = m21.corpus.chorales.Iterator()
bach_snippets = extract_snippets(corpus=bach_chorales)

print(bach_snippets)

In [ ]:
"""
here are the not-transposed outcomes, as comparison
"""

import music21 as m21
snippets = {}
bach_chorales = m21.corpus.chorales.Iterator()
for piece in bach_chorales:
    for part in piece.parts:
        if part.partName == "Soprano":
            id = piece.metadata.movementName
            soprano = part.flatten().notes
            notes = [note.pitch.nameWithOctave for note in soprano.notes][:10]
            snippets[id] = notes
snippets



In [19]:
"""
4. Dominant seventh chords
"""
import music21 as m21

# NOTAGEN

d7_count_ai = 0
d7_ai_chord_list = [] #do we want/need a list? maybe we could specify what d7 chords occur often?
for path in notagencorpus.getPaths():
    score = corpus.parse(path)
    scoreChords = score.chordify()
    for chord in scoreChords.recurse().getElementsByClass(m21.chord.Chord):
        if chord.isDominantSeventh():
            d7_ai_chord_list.append([chord.beatStr, chord])
            d7_count_ai += 1
        else:
            continue


# BACH
d7_count = 0
d7_bach_chord_list = []

bach_chorales = m21.corpus.chorales.Iterator()
for piece in bach_chorales:
    pieceChords = piece.chordify()
    for chord in pieceChords.recurse().getElementsByClass(m21.chord.Chord):
        if chord.isDominantSeventh():
            d7_bach_chord_list.append([chord.beatStr, chord])
            d7_count += 1
        else:
            continue

print(f"Total D7-Chord Count (Bach): {d7_count}")
print(f"Total D7-Chord Count (Notagen): {d7_count_ai}")






Total D7-Chord Count (Bach): 2831
Total D7-Chord Count (Notagen): 574


In [ ]:
"""
5. Check for parallel fifths
"""
# figuredBass.checker.parallelFifths?
